In [1]:
# Importing Libraries

import os
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import optimizers
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    matthews_corrcoef,
    cohen_kappa_score
)

In [2]:
# Paths of VGG, ResNET and Datasets

model_path = r"D:\Code of Malaria Detection\Ensemble\Models"
data_path = r"D:\Code of Malaria Detection\Ensemble"

vgg_file = os.path.join(model_path, "Malaria_VGG19_Model.h5")
resnet_file = os.path.join(model_path, "Malaria_ResNET50_Model.h5")

# Load models
model_vgg = load_model(vgg_file, compile=False)
model_res = load_model(resnet_file, compile=False)

# Compile models
optimizer = optimizers.SGD(learning_rate=0.0001, momentum=0.9)
model_vgg.compile(loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
model_res.compile(loss="categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])

In [3]:
# Load Test Datasets

X_Test = pickle.load(open(os.path.join(data_path, "X_Test_224.pickle"), "rb"))
Y_Test = pickle.load(open(os.path.join(data_path, "Y_Test_224.pickle"), "rb"))

# Ensemble weights
vgg_weight = 0.52
res_weight = 0.48

In [ ]:
# Model Prediction

pred_vgg = model_vgg.predict(X_Test, verbose=0)
pred_res = model_res.predict(X_Test, verbose=0)

# Weighted Average Ensemble

ensemble_pred = vgg_weight * pred_vgg + res_weight * pred_res
y_pred = np.argmax(ensemble_pred, axis=1)

# True labels

y_true = Y_Test.flatten()  # if Y_Test is one-hot, use np.argmax(Y_Test, axis=1)


In [ ]:
# Metrics

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
spec = tn / (tn + fp)
miss_rate = 1 - rec
fall_out = 1 - spec
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)

# Save Results

Type = "Malaria Detection"
with open(f"{Type}_Result_of_Ensemble.txt", "w") as f:
    f.write(f"Accuracy: {acc}\n")
    f.write(f"Precision: {prec}\n")
    f.write(f"Recall: {rec}\n")
    f.write(f"Specificity: {spec}\n")
    f.write(f"F1_Measure: {f1}\n")
    f.write(f"Miss_Rate: {miss_rate}\n")
    f.write(f"Fall_Out: {fall_out}\n")
    f.write(f"Matthews Correlation Coefficient: {mcc}\n")
    f.write(f"Cohen's kappa: {kappa}\n")

print("Evaluation complete. Results saved to file.")


In [ ]:
import matplotlib.pyplot as plt

# Pick an index of the test image

idx = 225   # change this number to see different test images

# Get the image and label

img = X_Test[idx]
true_label = Y_Test[idx]  # if one-hot, use np.argmax(Y_Test[idx])
pred_label = y_pred[idx]  # from ensemble predictions

# Plot

plt.imshow(img.astype("uint8"))  # if already in 0-255 format
# plt.imshow(img)  
# if scaled between 0-1
plt.title(f"True: {true_label}, Predicted: {pred_label}")
plt.axis()
plt.show()